# Validation of NEDAS using the Lorenz 1996 model case

## System setup (optional)

In [ ]:
%pip install NEDAS

## Load the NEDAS system and dependencies

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Image, display

from NEDAS.config import Config
from NEDAS import get_scheme

## Initialization

The config object holds all parameters defined in a yaml file, we can also make some changes

In [ ]:
config = Config(config_file="lorenz96/config.yml", quiet=True)

In [ ]:
obs_rec_id = 0
config.obs_def[obs_rec_id]

In [ ]:
config.obs_def[obs_rec_id]['err']['std'] = 0.1

In [ ]:
config.obs_def[obs_rec_id]['hroi'] = 5

In [ ]:
obs_x = [i for i in range(0, 20, 2)]
config.dataset_def['synthetic']['obs_x'] = obs_x
config.obs_def[obs_rec_id]['nobs'] = len(obs_x)

## Run the main scheme

The main scheme is the top level manager of runtime execution, it can be initialized with the config object.

Notes: 

- The first cycle takes a bit longer since the python functions are being compiled by numba.njit
- NEDAS always use real datetime to handle time controls. Lorenz96 is nondimensional, so we used a dt = 1 equivalent to 24 h convention.

In [ ]:
scheme = get_scheme(config)

In [ ]:
%rm -rf lorenz96/work

In [ ]:
scheme()

## Collect data from memory

In [ ]:
model = scheme.c.models['lorenz96']
dataset = scheme.c.datasets['synthetic']

In [ ]:
scheme.c.time = scheme.c.config.time_start

truth_state = []
while scheme.c.time < scheme.c.config.time_end:
    truth_state.append(model.read_var_from_memory(tag='truth', name='state', member=None, time=scheme.c.time))
    scheme.c.time = scheme.c.next_time


In [ ]:
scheme.c.time = scheme.c.config.time_start

prior_state = []
post_state = []
while scheme.c.time < scheme.c.config.time_end:
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var_from_memory(tag='prior', name='state', member=m, time=scheme.c.time))
    prior_state.append(ens)
    ens = []
    for m in range(scheme.c.nens):
        if scheme.c.time >= scheme.c.config.time_analysis_start and scheme.c.time <= scheme.c.config.time_analysis_end:
            ens.append(model.read_var_from_memory(tag='post', name='state', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(scheme.c.grid.x.shape, np.nan))
    post_state.append(ens)
    scheme.c.time = scheme.c.next_time


In [ ]:
nobs = scheme.c.obs.info.records[0].nobs
scheme.c.time = scheme.c.config.time_start

obs_val = []
obs_x = []
while scheme.c.time < scheme.c.config.time_end:
    if scheme.c.time >= scheme.c.config.time_analysis_start and scheme.c.time <= scheme.c.config.time_analysis_end:
        seq = dataset.read_obs_from_memory(tag='raw', name='state', member=None, time=scheme.c.time)
        obs_val.append(seq['obs'])
        obs_x.append(seq['x'])
    else:
        obs_val.append(np.full(nobs, np.nan))
        obs_x.append(np.full(nobs, np.nan))
    scheme.c.time = scheme.c.next_time

In [ ]:
plt.figure(figsize=(12,3))
plt.contourf(np.array(truth_state)[0:500,:].T)
#m = 1
#plt.contourf((np.array(prior_state)[:, m, :] - np.array(truth_state)).T)

## Some diagnostics

### Animation of the state over time

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ensemble_lines = [ax.plot([], [], color='c', alpha=0.5)[0] for _ in range(scheme.c.nens)]
ensemble_mean = ax.plot([], [], color='b', linewidth=2)[0]
obs_plot = ax.plot([], [], 'x', color='r', markersize=10)[0]
truth_line = ax.plot([], [], color='k', linewidth=2)[0]

def init():
    """Set up the axis limits and background."""
    ax.set_xlim(scheme.c.grid.x.min(), scheme.c.grid.x.max())
    ax.set_ylim(-10, 15)
    return ensemble_lines + [ensemble_mean, obs_plot, truth_line]

def update(n):
    """Update the data for frame n."""
    # Update ensemble
    for m in range(scheme.c.nens):
        ensemble_lines[m].set_data(scheme.c.grid.x, post_state[n][m])
    ensemble_mean.set_data(scheme.c.grid.x, np.mean(np.array(post_state[n]), axis=0))

    # Update observations
    obs_plot.set_data(obs_x[n], obs_val[n])
    
    # Update truth
    truth_line.set_data(scheme.c.grid.x, truth_state[n])
    
    ax.set_title(f"Time: {n*0.25:6.2f}")
    return ensemble_lines + [ensemble_mean, obs_plot, truth_line]

# 2. Create the animation
nt = len(truth_state)
ani = FuncAnimation(fig, update, frames=range(nt), init_func=init, blit=True, interval=50)
plt.close()

HTML(ani.to_jshtml())

### Time series of error and spread

## Experiment with different parameters

Reproducing Fig. 5.7 of [Boquet's lecture notes](https://cerea.enpc.fr/HomePages/bocquet/teaching/assim-mb-en-0.52.pdf)

In [ ]:
nens_try = [5, 6, 7, 8, 9, 10, 20, 30, 40, 50]
#F_try = [8, 10]


## A few details to look at

In [ ]:
# Visualize the localization functions
pos = 1.2
hroi = 10

dist = scheme.c.grid.distance(scheme.c.grid.x, pos)
rho = scheme.c.localization_funcs['horizontal'](dist, hroi)

plt.figure(figsize=(6,3))
plt.plot(rho)